# Schedules to Delta table

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCOUNT    = "footballmanagerli"
SRC        = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/schedules/"
DEST       = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/schedules/schedules"
CHECKPOINT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/schedules"
TABLE      = "nfl.schedules"

In [ ]:
schedule_schema = StructType([
    StructField("id",           StringType()),
    StructField("date",         StringType()),
    StructField("week",         IntegerType()),
    StructField("status",       StringType()),
    StructField("completed",    BooleanType()),
    StructField("homeTeamId",   StringType()),
    StructField("awayTeamId",   StringType()),
    StructField("homeScore",    IntegerType()),
    StructField("awayScore",    IntegerType()),
    StructField("venue",        StringType()),
])

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nfl")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    id            STRING    COMMENT 'Game ID (primary key)',
    date          STRING    COMMENT 'Game date (ISO 8601)',
    week          INT       COMMENT 'NFL week number',
    status        STRING    COMMENT 'Game status (e.g., Final, In Progress)',
    completed     BOOLEAN,
    homeTeamId    STRING,
    awayTeamId    STRING,
    homeScore     INT,
    awayScore     INT,
    venue         STRING,
    snapshot_at   TIMESTAMP COMMENT 'When the snapshot file was written',
    source_file   STRING,
    ingested_at   TIMESTAMP
)
USING DELTA
LOCATION '{DEST}'
""")

In [ ]:
raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .schema(schedule_schema)
    .load(SRC)
    .select(
        "*",
        F.col("_metadata.file_path").alias("source_file"),
        F.col("_metadata.file_modification_time").alias("snapshot_at"),
    ))

schedules = (raw
    .select(
        F.col("id"),
        F.col("date"),
        F.col("week"),
        F.col("status"),
        F.col("completed"),
        F.col("homeTeamId"),
        F.col("awayTeamId"),
        F.col("homeScore"),
        F.col("awayScore"),
        F.col("venue"),
        "snapshot_at",
        "source_file",
        F.current_timestamp().alias("ingested_at"),
    ))

In [ ]:
def upsert_batch(batch_df, batch_id):
    # Dedupe within the batch: keep the newest snapshot per game
    w = Window.partitionBy("id").orderBy(F.col("snapshot_at").desc())
    latest = (batch_df
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn"))

    # Merge into the table: insert new games, update if snapshot is newer
    (DeltaTable.forName(spark, TABLE).alias("t")
        .merge(latest.alias("s"), "t.id = s.id")
        .whenMatchedUpdateAll(condition="s.snapshot_at > t.snapshot_at")
        .whenNotMatchedInsertAll()
        .execute())

(schedules.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

In [ ]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT id) AS unique_games FROM nfl.schedules;

SELECT date, week, homeTeamId, awayTeamId, completed FROM nfl.schedules ORDER BY date DESC LIMIT 20;